10.5 一个完整的卷积神经网络

之前我们学过了全连接层、卷积层、池化层，这一节我们就把这些模块组合起来实现一个完整的卷积神经网络（ Convolutional Neural Network，CNN）。

如何进行图像分类

之前我们讲了卷积层和池化层，它们的输出都是带有空间位置信息的特征图，但是这些特征信息如何用来进行图像分类呢？

在进行了足够的卷积层后，得到的特征图都是具有概括性的高级特征了。我们只需要把多通道的高级特征图展开，连成一个一维向量，后边再接全连接层进行分类就可以了。

想想我们之前用全连接神经网络进行手写数字识别。因为每个图片只有一个数字，并且数字居中。没有利用到局部性和平移不变性，所以全连接网络就可以很好的识别。现在有了卷积层提取高级特征，后边用全连接层在高级特征（而不是像素值）上进行分类任务，在更通用的图像分类任务上会取得更好的效果。

各个模块的作用

卷积层

一个卷积层由多个Filter构成，每个Filter利用图像问题的局部性和平移不变性，用滑动窗口对局部多通道特征进行融合计算，输出一个新的特征。每个Filter输出的新特征构成一个通道。多个Filter构成多个通道。

激活函数

卷积层输出的特征需要经过激活函数来进行激活，带来非线性。最常使用的激活函数就是ReLU。

池化层

一个或者多个卷积层后，一般会通过一个最大池化层来提取最强特征，抑制其他特征。并且缩小特征图尺寸，缩写特征图的操作也被叫做下采样（Subsampling）。

Flatten

将特征图展平为一维向量。

全连接层

卷积操作完成后，最后用全连接进行汇总，完成分类/回归任务。

Dropout

防止过拟合，一般应用在全连接层，如果你要应用在卷积层输出的特征图，注意需要将多个通道的同一位置一起丢弃（Spatial Dropout）。

组合多个模块
假设我们输入的还是之前手写数字识别的28x28的灰度图像。

输入：28x28x1
C1，卷积层：8个3x3的卷积核，Padding=1，Stride=1，输出28x28x8的特征图。ReLU激活。
P2，最大池化层：池化核2x2，步长为2，输出为14x14x8的特征图。
C3，卷积层：16个3x3的卷积核，Padding=1，Stride=1，输出14x14x16的特征图。ReLU激活。
P4，最大池化层，池化核2x2，步长为2，输出为7x7x16的特征图。
C5，卷积层：32个3x3的卷积核，Padding=0，Stride=1，输出5x5x32的特征图。ReLU激活。Flatten展开为1x800的特征。
F6，全连接层：输入特征800，输出特征64，ReLU激活。
输出层，输入特征64，输出特征10，Softmax激活。
接下来我们计算一下每层的参数： 对于C1层，每个Filter的参数等于卷积核的3x3x1，其中1是输入通道数。加上bias的一个参数，一共10个参数，8个Filter，一共80个参数。

池化层没有参数。

C3层，每个Filter的参数等于卷积核的3x3x8，其中8是输入通道数。加上bias的一个参数，一共73个参数，16个Filter，一共1168个参数。

C5层，每个Filter的参数为(3x3x16+1)x32=4640个参数。

F6层，800x64+64=51264个参数。

输出层，64x10+10=650个参数。

可以看到参数量最大的就是全连接层，它占了所有参数的88%。

通过观察上边的网络结构图，可以发现随着网络深度的增加，特征图的尺寸在逐渐缩小，通道数在逐步增加。以人脸识别为例，浅层每个特征的感受野小，每个通道需要更多的特征去编码一些低级的局部特征（边缘，纹理）。深层随着每个特征感受野的增加，一个通道只需要少量的特征就可以编码更高级的全局语义（五官，表情）。通道数的增加使得网络可以学到更多的潜在高级特征。比如：微笑的眼睛，生气的嘴巴等。

换句话说，低级特征每个特征感受野小，对位置敏感，所以需要更大的特征图来精确定位位置，但是通道数少。高级全局特征每个特征感受野大，对位置不那么敏感，所以通道数多，特征图尺寸小。

10.6

1乘1卷积和全局平均池化层

在我们上一节的例子里，我们发现在整个卷积神经网络里参数量最大的是全连接层。它的参数量是800x64，其中800是因为每个特征图由25个特征，一共32个通道，25x32=800。作为一个全连接层它在整个网路参数里占据了过大的比重，今天我们就来学习两种方法，1乘1卷积和全局平均池化层，利用它们都可以降低全连接层的参数。

1乘1卷积初听起来好像没什么用，因为对于图像，多要通过多个像素相互对比才能表达信息。如果只有一个像素，它表达不了任何信息。

比如对于上图左边的特征图，经过一个1x1的卷积核，weight为2，经过计算就是对每个位置的特征放大2倍。看起来没什么用。这是因为这个例子里的输入通道是1，但实际上1x1卷积是用在多通道上的。

比如对于一个2x2x4的输入特征图，用一个1x1的卷积核进行卷积操作：

如下图所示，比如对输入特征图每个通道的左上角特征进行1x1卷积操作，就等于对每个通道左上角特征进行了一个全连接操作：

通过上图可以看到1x1卷积可以看成是一个全连接的神经元，它通过共享参数对多个通道相同位置的特征进行融合，线性回归计算，并通过激活函数，产生新的特征。所以它有融合特征，获取新特征的能力。1x1卷积不会加Padding，步长为1，生成的特征图尺寸和输入特征图一致。

1x1卷积最重要的作用是改变特征图的通道数。如果输入特征图的维度为W，H，C1，定义C2个1x1卷积Filter，就可以生成维度为W，H，C2的特征图了。

比如上一节例子中，卷积层最后的输出为5x5x32。如果我们定义4个1x1的卷积Filter，那么就可以生成一个5x5x4的特征图。这个特征图接全连接层的话，全连接层的参数就为：100x64，全连接层的参数量就只有原来的八分之一了。

一般1x1卷积只会用到网络深层，对已经通过普通卷积提取的特征进行融合，不会直接用在原始图片像素上。

全局平均池化层

全局平均池化（Global Average Pooling, GAP）是一种特殊的池化操作，它对每个通道的整个空间维度进行平均运算，将 W×H×C 的特征图压缩为 1×1×C。例如，对一张大小为 5×5×32 的特征图使用全局平均池化，最终会得到一个 1×1×32 的输出。并且它不会额外引入参数。对于上一节的例子，如果我们对卷积最后的特征图先进行全局平均池化，再接全连接层，那么全连接层的参数量就是32×64，大小是原来的25分之一。

两者配合使用

在实际中，有一种更高级的用法，那就是1乘1卷积配合全局平均池化层一起使用。 还是以上一节对手写数字分类的例子，最后卷积层输出的特征图大小为5×5×32，先经过10个1×1卷积，将特征图调整为5×5×10。这里之所以调整通道数为10，是因为我们最终要分的类别数是10。然后对5×5×10的特征图接GAP，得到1×1×10，最后接Flatten，再接softmax，直接对数字0-9进行分类。

可以看到，通过1乘1卷积和全局平均池化层，可以完全省去全连接层。大大减少模型参数。

去掉全连接层的好处

在卷积神经网络里去掉全连接层的好处除了上边所说的减少参数量外，还有一个非常好的特性，就是输入可以是任意大小的图片了。下边我们来详细解释。

假如输入图片是一个6×6×1的灰度图片，经过一个3×3的卷积操作，padding为1，stride为1，输出特征还是6×6×1。如果后边我要接一个全连接层，输入就是36，输出为8。那么这个全连接层的参数就是36×8。因为全连接层的参数需要定义网络时就给定，所以它就反向固定了输入图像的尺寸必须为6×6。 假如我们现在有一个7×7×1的输入图片，卷积操作对图像大小没有要求，因为不论图片多大，它都工作在3x3的区域上。图片大，就多计算几次。经过卷积操作后，输出特征还是7×7×1，这时就没有办法接全连接层了，因为这时输入成了49了。而我们定义的全连接层的输入是36。

同样1×1卷积操作和GAP操作都对图像尺寸没有限定，所以去掉全连接层的卷积神经网络更加灵活。